# Train independent JSBSim F-16 pilots with PPO

Train any number of independent pilots with the real `PPOTrainer` and repository-pinned
BVR Sim/JSBSim backend. Each pilot has its own
network, optimizer, seed, reward function, MLflow run, and checkpoint directory.
Pilots train sequentially; simulator workers still collect each pilot's flights concurrently.

The transformer sees two seconds of 10 Hz observations and separate energy estimates for
both aircraft. Each flight remains a one-versus-one engagement against the existing baseline.
Pilot names are identifiers, not fixed roles. Each pilot can use an arbitrary reward factory.

The batch writes `pilots.json` plus one directory per pilot under a new timestamped
`artifacts/rl_pilot_notebook/` directory. Existing batches are never overwritten.

## Setup

Install dependencies from the repository root, then restart the kernel:

```bash
python -m pip install -e '.[ml,video]'
```

For GPU training, install a CUDA-enabled PyTorch build in the selected kernel's
environment and restart the kernel. See `docs/multi_pilot_training.md` for setup.
Policy inference and PPO updates use CUDA when available; JSBSim and opponent
control run in separate CPU processes so Python control code can use multiple cores.
Processes are reused across epochs. `BVR_PILOT_DEVICE` can explicitly select a device.
After a code update, restart the kernel and run from the top; confirm the configuration
prints `simulator_executor: process`. The first flight batch includes worker startup.
`BVR_PILOT_EXECUTOR=thread` restores the previous collector for a timing comparison.

For an installation check, set `BVR_PILOT_EPOCHS=1`, `BVR_PILOT_EPISODE_SECONDS=1`,
`BVR_PILOT_SCENARIOS=3`, `BVR_PILOT_EVALUATION_SCENARIOS=3`, and
`BVR_PILOT_UPDATE_EPOCHS=1` before starting Jupyter. Set `BVR_PILOT_COUNT` to any positive
number (default 10), or define an explicit `PILOTS` list below.
To leave capacity for VS Code and other work, `BVR_PILOT_WORKERS` defaults to 80%
of the logical CPU count, rounded down with a minimum of one worker. `BVR_PILOT_TORCH_THREADS`
caps PyTorch CPU operations at up to 8 threads by default. Both are configurable
before setup; explicit environment values override the defaults. These limits
reduce concurrency but do not reserve specific cores or guarantee a CPU percentage.
CPU load falls during GPU updates. Compare `rollout_seconds` and `update_seconds` in
the saved metrics; `simulator_effective_cores` measures CPU use during simulator steps.

In [ ]:
import json
import os
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Leave CPU capacity for the editor and other applications.
DEFAULT_SIMULATOR_WORKERS = max(1, ((os.cpu_count() or 1) * 80) // 100)
TORCH_THREADS = int(os.getenv(
    "BVR_PILOT_TORCH_THREADS", str(min(8, DEFAULT_SIMULATOR_WORKERS))
))
if TORCH_THREADS < 1:
    raise ValueError("BVR_PILOT_TORCH_THREADS must be positive")
torch.set_num_threads(TORCH_THREADS)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "bvr_behavior_prediction").exists():
    REPO_ROOT = REPO_ROOT.parent
for source_root in (REPO_ROOT, REPO_ROOT / "bvr_sim_source"):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

from bvr_behavior_prediction.rl.bvr_jsbsim_backend import BVRJSBSimBackend
from bvr_behavior_prediction.rl.config import PilotTrainingConfig, RewardWeights
from bvr_behavior_prediction.rl.environment import BluePilotEnvironment
from bvr_behavior_prediction.rl.multi_pilot import PilotSpec, train_pilots, load_trained_pilot
from bvr_behavior_prediction.rl.reward import combat_reward_definition
from bvr_behavior_prediction.rl.trainer import PPOTrainer

SEED = int(os.getenv("BVR_PILOT_SEED", "7"))
evaluation_seed_value = os.getenv("BVR_PILOT_EVALUATION_SEED")
EVALUATION_SEED = int(evaluation_seed_value) if evaluation_seed_value else None
EPOCHS = int(os.getenv("BVR_PILOT_EPOCHS", "80"))
EPISODE_SECONDS = float(os.getenv("BVR_PILOT_EPISODE_SECONDS", "120"))
PILOT_COUNT = int(os.getenv("BVR_PILOT_COUNT", "10"))
if PILOT_COUNT < 1:
    raise ValueError("BVR_PILOT_COUNT must be positive")
BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUTPUT_DIR = Path(os.getenv(
    "BVR_PILOT_OUTPUT", REPO_ROOT / "artifacts/rl_pilot_notebook" / BATCH_ID
)).resolve()
DEVICE = os.getenv("BVR_PILOT_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
print({"device": DEVICE, "pilot_count": PILOT_COUNT, "epochs_per_pilot": EPOCHS,
       "torch_version": torch.__version__, "cuda_available": torch.cuda.is_available(),
       "torch_threads": torch.get_num_threads(), "output": str(OUTPUT_DIR)})

## BVR Sim backend with JSBSim flight dynamics

Each independent pilot controls one F-16 through the complete `SkillManager` catalogue.
The opponent uses BVR Sim's existing simple baseline. The adapter translates tactical
skills into flight controls and records ACMI telemetry. Logs are isolated by pilot.

In [ ]:
BASE_OBSERVATION_SIZE = BVRJSBSimBackend.OBSERVATION_SIZE
ENERGY_FEATURES = 4
OBSERVATION_SIZE = BASE_OBSERVATION_SIZE + ENERGY_FEATURES


def make_environment_factory(run_config):
    def bvr_jsbsim_backend_factory(scenario_dict, recording_path=None):
        return BVRJSBSimBackend(
            scenario_dict,
            recording_path=recording_path,
            log_dir=run_config.output_dir / "simulator_logs",
        )

    def make_environment(scenario, recording_path=None):
        return BluePilotEnvironment(
            bvr_jsbsim_backend_factory,
            scenario,
            planning_horizon_s=run_config.planning_horizon_s,
            episode_duration_s=run_config.episode_duration_s,
            history_duration_s=run_config.history_duration_s,
            sample_interval_s=run_config.sample_interval_s,
            recording_path=recording_path,
        )

    return make_environment

## Configure pilots and reward functions

Each pilot's training set-ups are generated once from its seed and reused across epochs.
Every pilot uses the same fixed validation geometries and simulator seeds, from a separate
seed range. This set is used for checkpoint selection; reserve another set for final tests.

`PILOTS` is an arbitrary-length experiment roster. Each entry has a unique ID, a seed,
and a `RewardDefinition`. The example below varies event weights continuously with pilot
index, so increasing the count does not cycle through a fixed set of named behaviours.
These weights are editable experiment examples, not validated behavioural categories.

Replace `make_reward_definition` with your own factory or write an explicit list of
`PilotSpec` entries. Different pilots may use entirely different reward formulas. Each
factory must construct a fresh callable that accepts simulator `info` and returns
`(reward, components)`. See `docs/multi_pilot_training.md` for a custom-formula example.
For a controlled reward comparison, pilots may share a seed; their models and optimizers
are still independent. Record all settings in the definition's `parameters` and version it.

In [ ]:
def make_reward_definition(pilot_index):
    # Example only: replace this function to define your own objectives.
    scale = 1.0 + 0.25 * pilot_index
    weights = replace(
        RewardWeights(),
        crashed=-100.0 * scale,
        shot_down=-150.0 * scale,
        incoming_missile_avoided=20.0 * scale,
    )
    return combat_reward_definition(f"objective_{pilot_index + 1:03d}", weights=weights)


PILOTS = [
    PilotSpec(
        pilot_id=f"pilot_{index + 1:03d}",
        seed=SEED + 1000 * index,
        reward=make_reward_definition(index),
    )
    for index in range(PILOT_COUNT)
]
display(pd.DataFrame([
    {"pilot_id": pilot.pilot_id, "seed": pilot.seed,
     "reward_name": pilot.reward.name, "parameters": pilot.reward.parameters}
    for pilot in PILOTS
]))

In [ ]:
config = PilotTrainingConfig(
    seed=SEED,
    evaluation_seed=EVALUATION_SEED,
    epochs=EPOCHS,
    episode_duration_s=EPISODE_SECONDS,
    planning_horizon_s=1.0,
    history_duration_s=2.0,
    sample_interval_s=0.1,
    transformer_heads=4,
    transformer_layers=2,
    scenarios_per_epoch=int(os.getenv("BVR_PILOT_SCENARIOS", "50")),
    evaluation_scenarios_per_epoch=int(os.getenv("BVR_PILOT_EVALUATION_SCENARIOS", "50")),
    simulator_workers=int(os.getenv("BVR_PILOT_WORKERS", str(DEFAULT_SIMULATOR_WORKERS))),
    simulator_executor=os.getenv("BVR_PILOT_EXECUTOR", "process"),
    hidden_size=64,
    learning_rate=3e-4,
    update_epochs=int(os.getenv("BVR_PILOT_UPDATE_EPOCHS", "50")),
    minibatch_size=64,
    checkpoint_interval=max(1, EPOCHS // 5),
    diagnostic_interval=1,
    output_dir=OUTPUT_DIR,
    mlflow_experiment="jsbsim-independent-pilots",
    mlflow_tracking_uri=f"sqlite:///{(REPO_ROOT / 'artifacts/mlflow.db').as_posix()}",
)
print(json.dumps(config.as_dict(), indent=2))

## Train the roster

This cell trains each pilot from scratch and saves its selected `best_model.pt`. The batch
refuses an existing output directory, so rerunning this cell cannot erase a previous result.
Run setup again for a new timestamp, or choose a new `BVR_PILOT_OUTPUT` directory.

Each checkpoint is selected with that pilot's own validation reward. After training, the
selected checkpoint is evaluated once with a **common benchmark reward**. Use
`benchmark_return` and outcome rates for cross-pilot comparisons; `selection_return` uses
different reward scales. More pilots multiply the total training work.

In [ ]:
results = train_pilots(
    pilots=PILOTS,
    config=config,
    env_factory_builder=make_environment_factory,
    observation_size=OBSERVATION_SIZE,
    device=DEVICE,
)
summary = pd.DataFrame(results)
display(summary[[
    "pilot_id", "seed", "reward_name", "selection_return", "benchmark_return",
    "survival_rate", "opponent_destroyed_rate", "checkpoint",
]])
print(f"Batch manifest: {OUTPUT_DIR / 'pilots.json'}")

In [ ]:
# Select any completed pilot for inspection. This does not rerun training.
SELECTED_PILOT = os.getenv("BVR_PILOT_VIEW", results[0]["pilot_id"])
selected_result = next(result for result in results if result["pilot_id"] == SELECTED_PILOT)
selected_spec = next(pilot for pilot in PILOTS if pilot.pilot_id == SELECTED_PILOT)
RUN_DIR = OUTPUT_DIR / selected_result["output_dir"]
selected_config = replace(
    config, seed=selected_spec.seed,
    evaluation_seed=selected_result["evaluation_seed"], output_dir=RUN_DIR,
)
trainer = PPOTrainer(
    make_environment_factory(selected_config), OBSERVATION_SIZE, selected_config,
    device=DEVICE, reward_definition=selected_spec.reward,
)
trainer.pilot.load_state_dict(load_trained_pilot(RUN_DIR, DEVICE).state_dict())
trained_pilot = trainer.pilot.eval()

FINAL_EVALUATION_DIR = RUN_DIR / "final_evaluation"
FINAL_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
final_evaluation_rollouts = []
final_evaluation_manifest = []
for index, scenario in enumerate(trainer.evaluation_scenarios):
    episode_seed = selected_config.resolved_evaluation_seed + index
    acmi_path = FINAL_EVALUATION_DIR / f"scenario_{index:03d}_seed_{episode_seed}.txt.acmi"
    rollout, score, final_info = trainer._episode(
        scenario, episode_seed, recording_path=acmi_path, deterministic=True,
    )
    final_evaluation_rollouts.append(rollout)
    final_evaluation_manifest.append({
        "pilot_id": SELECTED_PILOT,
        "checkpoint_sha256": selected_result["checkpoint_sha256"],
        "reward": selected_spec.reward.as_dict(),
        "scenario_index": index, "seed": episode_seed, "score": score,
        "scenario": scenario.as_dict(), "final_info": final_info, "acmi": acmi_path.name,
    })

manifest_path = FINAL_EVALUATION_DIR / "manifest.json"
manifest_path.write_text(json.dumps(final_evaluation_manifest, indent=2, default=str), encoding="utf8")
print(f"Recorded {len(final_evaluation_manifest)} replays for {SELECTED_PILOT}")
print(f"ACMI directory: {FINAL_EVALUATION_DIR}")

## Compare the pilots

The shared benchmark and survival/opponent-destroyed rates are comparable because all
pilots see the same validation scenarios, deterministic action selection, and benchmark
reward. These are validation results, not evidence of generalization to unseen agents.

Each pilot also has its own training curves. Their reward totals have different scales,
so use them to assess progress within a pilot. Evaluation occurs on the first, last, and
every fifth epoch by default; only fresh evaluation points are plotted. Loss need not fall
monotonically. The JSONL files and `pilots.json` can be reloaded later without retraining.

In [ ]:
batch_manifest = json.loads((OUTPUT_DIR / "pilots.json").read_text(encoding="utf8"))
completed = [pilot for pilot in batch_manifest["pilots"] if pilot["status"] == "completed"]
summary = pd.DataFrame(completed)
metrics_by_pilot = {
    pilot["pilot_id"]: pd.read_json(
        OUTPUT_DIR / pilot["output_dir"] / "training_metrics.jsonl", lines=True
    )
    for pilot in completed
}
display(summary[["pilot_id", "benchmark_return", "benchmark_return_std",
                 "survival_rate", "opponent_destroyed_rate"]])

In [ ]:
figure, axes = plt.subplots(len(completed), 2, figsize=(12, 4 * len(completed)), squeeze=False)
for row, pilot in enumerate(completed):
    metrics = metrics_by_pilot[pilot["pilot_id"]]
    evaluated = metrics[metrics["evaluation_performed"] == 1]
    axes[row, 0].plot(evaluated["epoch"], evaluated["mean_return"], marker="o", label="Validation")
    axes[row, 0].plot(metrics["epoch"], metrics["training_mean_return"], alpha=0.4, label="Training")
    axes[row, 0].plot(metrics["epoch"], metrics["best_return"], linestyle="--", label="Best")
    axes[row, 0].set(title=f"{pilot['pilot_id']}: own reward", xlabel="Epoch", ylabel="Return")
    axes[row, 0].legend()
    axes[row, 1].plot(metrics["epoch"], metrics["loss"], color="tab:orange")
    axes[row, 1].set(title=f"{pilot['pilot_id']}: PPO loss", xlabel="Epoch", ylabel="Loss")
    for axis in axes[row]:
        axis.grid(alpha=0.25)
figure.tight_layout()
plt.show()

figure, axis = plt.subplots(figsize=(8, 4))
axis.bar(summary["pilot_id"], summary["benchmark_return"],
         yerr=summary["benchmark_return_std"], capsize=5)
axis.set(title="Selected checkpoints on the common benchmark",
         xlabel="Pilot", ylabel="Return (error bars: scenario standard deviation)")
figure.tight_layout()
plt.show()

## Inspect the selected pilot

The replay cell saves the selected pilot's checkpoint hash and reward definition alongside
every ACMI flight. Change `BVR_PILOT_VIEW` or `SELECTED_PILOT` in that cell to inspect
another completed pilot. Open the `.txt.acmi` files in Tacview for the full flights.

In [ ]:
rollout = final_evaluation_rollouts[0]
first_evaluation = final_evaluation_manifest[0]
evaluation_return = first_evaluation["score"]
final_info = first_evaluation["final_info"]
print(f"Example ACMI: {FINAL_EVALUATION_DIR / first_evaluation['acmi']}")
selected_skills = [trained_pilot.skill_names[index] for index in rollout.skills]
skill_counts = pd.Series(selected_skills).value_counts().rename_axis("skill").rename("decisions")
print(f"Evaluation return: {evaluation_return:.2f}")
display(skill_counts.to_frame())

skill_counts.sort_values().plot.barh(figsize=(8, max(3, len(skill_counts) * 0.35)))
plt.title("Skills selected in the first fixed evaluation rollout")
plt.xlabel("Decisions")
plt.tight_layout()
plt.show()


## Next steps

- Compare the trained pilots in Tacview and on a separate held-out scenario set.
- Tune each `RewardDefinition`, or add more `PilotSpec` entries with unique IDs.
- Use `load_trained_pilot(run_directory)` to load a checkpoint after restarting Python.
- Retain `config.json`, `manifest.json`, and the checkpoint together when publishing through
  `shared/models/`. Each generated flight corpus should identify its pilot and checkpoint hash.
- Checkpoints contain inference weights; they do not resume an interrupted optimizer.